# 🐍 Dia 4 — Funções Avançadas & Comprehensions em Python

**Autor:** Gutemberg Rapôso · **Data:** 2026-09-20 · **Fase 1 — Python Basics (Dia 4/28)**

> Parte de uma jornada estruturada de 210 dias: *Junior → Senior Data Scientist | Aberto a Oportunidades Remotas*.

## 🎯 Objetivos de Aprendizagem

Ao final deste notebook, você será capaz de:

1. Escrever **funções** robustas com *type hints*, *docstrings*, `*args` e `**kwargs`.
2. Usar **funções lambda** para transformações rápidas e ordenação customizada.
3. Aplicar **`map`, `filter` e `functools.reduce`** em dados reais.
4. Dominar **list, dict e set comprehensions** (inclusive aninhadas e com condição).
5. Criar **generators** e entender a economia de memória frente às listas.
6. Usar **`functools.partial`** para montar pipelines de Data Science reutilizáveis.
7. Aplicar programação funcional ao **processamento de texto / PLN** — conectando à minha
   formação em **Linguística e Sociolinguística**.
8. Comparar **desempenho** entre `for`, comprehensions e `map` com `timeit`.

> 📌 **Convenção:** as *notas de estudo* estão em **português (PT-BR)**; o **código, comentários
> e docstrings** estão em **inglês**, como é padrão em projetos profissionais de dados.

## 🧠 Por que Programação Funcional importa para Data Science?

A programação funcional trata **funções como cidadãos de primeira classe**: elas podem ser passadas
como argumento, retornadas e compostas. Em Data Science isso é valioso porque:

- **Transformações declarativas** — descrevemos *o que* queremos (ex.: "todos os preços em reais"),
  não *como* iterar. Isso torna o código mais legível e menos propenso a bugs.
- **Menos estado mutável** — comprehensions e `map`/`filter` evitam variáveis auxiliares e efeitos
  colaterais, facilitando a depuração e a paralelização.
- **Pipelines reutilizáveis** — com `partial` e composição de funções montamos etapas de
  pré-processamento encadeáveis (muito comum em `sklearn`, `pandas.apply`, PySpark).
- **Eficiência** — comprehensions e generators costumam ser mais rápidos e econômicos em memória do
  que laços `for` explícitos.

Em resumo: dominar essas ferramentas deixa seu código **mais limpo, mais rápido e mais profissional** —
exatamente o que se espera de um Cientista de Dados pleno/sênior.

In [ ]:
# Standard scientific + functional-programming toolkit
import pandas as pd
import seaborn as sns

# Functional / iteration helpers from the standard library
from functools import reduce, partial
import itertools
from collections import Counter, defaultdict

import re
import sys
import timeit

# Display options for cleaner DataFrame output
pd.set_option("display.max_rows", 12)
pd.set_option("display.width", 100)

print("pandas :", pd.__version__)
print("seaborn:", sns.__version__)
print("Environment ready ✅")

## 1️⃣ Revisão de Funções em Python

Uma boa função é **pequena, previsível e documentada**. Em projetos profissionais usamos:

- **Type hints** (`def f(x: int) -> int:`) — comunicam a intenção e habilitam checagem estática.
- **Docstrings** — explicam propósito, parâmetros e retorno (padrão *NumPy* ou *Google*).
- **`*args` / `**kwargs`** — permitem funções flexíveis que aceitam número variável de argumentos.

Vamos revisar cada um desses pontos.

In [ ]:
def normalize_score(score: float, minimum: float = 0.0, maximum: float = 100.0) -> float:
    '''Scale a raw score to the [0, 1] range using min-max normalization.

    Parameters
    ----------
    score : float
        The raw value to normalize.
    minimum, maximum : float
        The bounds of the original scale (default 0-100).

    Returns
    -------
    float
        The normalized value, clipped to [0, 1].
    '''
    if maximum == minimum:
        raise ValueError("maximum and minimum must differ")
    scaled = (score - minimum) / (maximum - minimum)
    return max(0.0, min(1.0, scaled))  # clip to [0, 1]


# Quick sanity checks
print(normalize_score(50))          # -> 0.5
print(normalize_score(90, 0, 120))  # -> 0.75
print(normalize_score(150))         # clipped -> 1.0
print(normalize_score.__doc__.splitlines()[0])

In [ ]:
def describe_sample(*values: float, **options: str) -> dict:
    '''Return simple descriptive stats for any number of numeric values.

    *values  -> variable positional numbers
    **options -> arbitrary metadata (e.g. label='english_exam')
    '''
    stats = {
        "n": len(values),
        "mean": sum(values) / len(values) if values else float("nan"),
        "min": min(values) if values else None,
        "max": max(values) if values else None,
    }
    stats.update(options)  # merge metadata coming from **kwargs
    return stats


result = describe_sample(7.5, 8.0, 9.2, 6.8, label="english_exam", cohort="2026")
for key, value in result.items():
    print(f"{key:>7}: {value}")

## 2️⃣ Funções Lambda (funções anônimas)

Uma **lambda** é uma função de uma linha, sem nome, definida com a palavra-chave `lambda`:

```python
lambda argumentos: expressão
```

Elas são ideais para **funções curtas e descartáveis**, especialmente como argumento de `sorted`,
`map`, `filter` e `pandas.apply`. Para lógica maior, prefira `def` (mais legível e testável).

In [ ]:
# A lambda is just a function object stored in a variable
square = lambda x: x ** 2
print("square(9) =", square(9))

# Real use case: custom sorting keys
linguists = [
    ("Saussure", 1857),
    ("Chomsky", 1928),
    ("Labov", 1927),
    ("Halliday", 1925),
]

by_birth_year = sorted(linguists, key=lambda person: person[1])
by_name_length = sorted(linguists, key=lambda person: len(person[0]))

print("By birth year :", by_birth_year)
print("By name length:", by_name_length)

In [ ]:
# Load the real seaborn 'tips' dataset (restaurant bills & tips)
tips = sns.load_dataset("tips")
print(tips.head())

# Use a lambda inside .apply() to engineer a new feature: tip percentage
tips["tip_pct"] = tips.apply(lambda row: round(row["tip"] / row["total_bill"] * 100, 2), axis=1)

# Classify each bill size with a lambda on a single column
tips["bill_size"] = tips["total_bill"].apply(lambda x: "high" if x >= 20 else "low")

print(tips[["total_bill", "tip", "tip_pct", "bill_size"]].head())

## 3️⃣ `map`, `filter` e `functools.reduce`

Este é o trio clássico da programação funcional:

| Função | O que faz | Retorna |
|--------|-----------|---------|
| `map(f, it)` | aplica `f` a **cada** elemento | iterador |
| `filter(f, it)` | mantém elementos onde `f(x)` é `True` | iterador |
| `reduce(f, it)` | **acumula** os elementos em um único valor | valor único |

⚠️ `map` e `filter` retornam **iteradores preguiçosos** — envolva com `list()` para materializar.

In [ ]:
# Real data: total bills from the tips dataset
bills = tips["total_bill"].tolist()

# map(): convert every bill from US$ to R$ (approx. rate) — returns a lazy iterator
usd_to_brl = 5.15
bills_brl = list(map(lambda b: round(b * usd_to_brl, 2), bills))

print("First 5 bills (US$):", [round(b, 2) for b in bills[:5]])
print("First 5 bills (R$) :", bills_brl[:5])

# map over multiple iterables in parallel: bill + tip -> grand total
totals = list(map(lambda bill, tip: round(bill + tip, 2),
                  tips["total_bill"], tips["tip"]))
print("First 5 grand totals:", totals[:5])

In [ ]:
# filter(): keep only the generous tips (>= 20% of the bill)
generous = list(filter(lambda pct: pct >= 20, tips["tip_pct"]))
print(f"Generous tips (>=20%): {len(generous)} of {len(tips)} bills")

# Combine filter + a named predicate for readability
def is_weekend(day: str) -> bool:
    '''Return True for Saturday/Sunday labels used in the tips dataset.'''
    return day in {"Sat", "Sun"}

weekend_days = list(filter(is_weekend, tips["day"]))
print("Weekend service records:", len(weekend_days))

In [ ]:
# reduce(): fold a sequence into a single accumulated value
total_revenue = reduce(lambda acc, b: acc + b, bills, 0.0)
print(f"Total revenue (reduce): US$ {total_revenue:,.2f}")
print(f"Cross-check with sum() : US$ {sum(bills):,.2f}")

# reduce can build non-numeric results too: find the largest bill
biggest_bill = reduce(lambda a, b: a if a > b else b, bills)
print(f"Biggest single bill    : US$ {biggest_bill:,.2f}")

## 4️⃣ List Comprehensions

Uma **list comprehension** cria listas de forma concisa e legível:

```python
[expressão for item in iterável if condição]
```

Vantagens sobre o `for` tradicional:
- **Mais curta e expressiva** (uma linha).
- Geralmente **mais rápida** (otimizada em C internamente).
- Deixa clara a *intenção* de "construir uma nova lista".

In [ ]:
numbers = list(range(1, 11))

# Classic for-loop approach
squares_loop = []
for n in numbers:
    squares_loop.append(n ** 2)

# Equivalent list comprehension
squares_comp = [n ** 2 for n in numbers]

print("for-loop     :", squares_loop)
print("comprehension:", squares_comp)
print("Identical?   :", squares_loop == squares_comp)

In [ ]:
# Filtering inside a comprehension: keep only high tip percentages
high_tips = [round(pct, 1) for pct in tips["tip_pct"] if pct >= 25]
print("Tips >= 25%:", sorted(high_tips, reverse=True)[:10])

# if/else must go BEFORE the 'for' (it is a ternary expression, not a filter)
labels = ["generous" if pct >= 20 else "standard" for pct in tips["tip_pct"]]
print("First 10 labels:", labels[:10])
print("Counts:", Counter(labels))

In [ ]:
# Nested comprehension: build a small multiplication grid (matrix)
matrix = [[row * col for col in range(1, 6)] for row in range(1, 6)]
for line in matrix:
    print(line)

# Flatten a nested list with a nested comprehension
flat = [value for line in matrix for value in line]
print("\\nFlattened length:", len(flat), "-> first 10:", flat[:10])

## 5️⃣ Dict e Set Comprehensions

A mesma sintaxe funciona para **dicionários** e **conjuntos**:

```python
{chave: valor for item in iterável}   # dict comprehension
{expressão for item in iterável}      # set comprehension (elimina duplicatas)
```

Úteis para construir mapeamentos (ex.: `palavra -> índice`) e extrair valores únicos.

In [ ]:
# Average tip percentage per weekday -> {day: mean_tip_pct}
avg_tip_by_day = {
    day: round(tips.loc[tips["day"] == day, "tip_pct"].mean(), 2)
    for day in tips["day"].unique()
}
print("Average tip % per day:", avg_tip_by_day)

# Build a word -> index vocabulary map (very common in NLP)
words = ["language", "data", "science", "language", "model"]
vocab = {word: idx for idx, word in enumerate(sorted(set(words)))}
print("Vocabulary index map:", vocab)

In [ ]:
# Set comprehension automatically removes duplicates
unique_party_sizes = {size for size in tips["size"]}
print("Unique party sizes:", sorted(unique_party_sizes))

# Distinct lowercased tokens from a sentence
sentence = "The cat sat on the mat and the cat slept"
unique_tokens = {token.lower() for token in sentence.split()}
print("Unique tokens:", sorted(unique_tokens))

## 6️⃣ Generators — eficiência de memória

Um **generator** produz valores **sob demanda** (*lazy evaluation*), sem armazenar tudo na memória.
Criamos generators de duas formas:

1. **Função com `yield`** — pausa e retoma a execução a cada `next()`.
2. **Generator expression** — igual à list comprehension, mas com **parênteses** `()`.

São essenciais para processar **grandes volumes de dados** (arquivos enormes, streams de texto) sem
estourar a RAM — cenário comum ao trabalhar com grandes corpora linguísticos.

In [ ]:
def running_average(values):
    '''Yield the running (cumulative) average of a numeric stream.

    Because it uses `yield`, only one partial sum is kept in memory at a time.
    '''
    total = 0.0
    for i, value in enumerate(values, start=1):
        total += value
        yield total / i


# Consume the generator lazily
gen = running_average(tips["total_bill"])
first_five = [round(next(gen), 2) for _ in range(5)]
print("Running average of bills (first 5 steps):", first_five)

In [ ]:
# Same logic, two containers: list (eager) vs generator (lazy)
list_version = [x ** 2 for x in range(100_000)]     # builds all 100k items now
gen_version = (x ** 2 for x in range(100_000))       # builds nothing yet

print("List object size :", sys.getsizeof(list_version), "bytes")
print("Generator size   :", sys.getsizeof(gen_version), "bytes")
print("Memory ratio     : ~{:.0f}x smaller".format(
    sys.getsizeof(list_version) / sys.getsizeof(gen_version)))

# The generator still produces correct results, one item at a time
print("Sum via generator:", sum(x ** 2 for x in range(100_000)))

## 7️⃣ `functools.partial` — pré-configurando funções

`partial` cria uma **nova função com alguns argumentos já fixados**. É perfeito para montar etapas de
um pipeline de Data Science reutilizáveis, evitando repetir os mesmos parâmetros a cada chamada.

In [ ]:
def convert_currency(amount: float, rate: float, symbol: str = "R$") -> str:
    '''Convert `amount` by `rate` and format it with a currency symbol.'''
    return f"{symbol} {amount * rate:,.2f}"


# Pre-configure specialized converters from one generic function
to_brl = partial(convert_currency, rate=5.15, symbol="R$")
to_eur = partial(convert_currency, rate=0.92, symbol="€")

print("100 USD ->", to_brl(100))
print("100 USD ->", to_eur(100))

# partial shines with map(): apply the pre-configured converter to a column
tips_in_brl = list(map(to_brl, tips["total_bill"].head()))
print("Bills in BRL:", tips_in_brl)

## 8️⃣ Aplicação em Texto / PLN — a ponte com a Linguística 🗣️

Aqui está o **diferencial** deste portfólio: aplicar programação funcional ao **processamento de
linguagem natural**. Como linguista, sei que texto bruto precisa ser **limpo, tokenizado e
quantificado** antes de qualquer análise.

Vamos usar um trecho de **domínio público** — a abertura de *Pride and Prejudice* (Jane Austen, 1813) —
e processá-lo com `Counter`, comprehensions, `map`, `filter` e `zip`.

In [ ]:
# Public-domain corpus excerpt: opening lines of "Pride and Prejudice" (Jane Austen, 1813)
CORPUS = '''It is a truth universally acknowledged, that a single man in possession
of a good fortune, must be in want of a wife. However little known the feelings or views
of such a man may be on his first entering a neighbourhood, this truth is so well fixed in
the minds of the surrounding families, that he is considered the rightful property of some
one or other of their daughters.'''

# Tokenize into lowercase word tokens using a regex + comprehension
tokens = [word for word in re.findall(r"[a-z']+", CORPUS.lower())]
print(f"Total tokens: {len(tokens)}")

# Word frequency with collections.Counter
freq = Counter(tokens)
print("Top 8 most common words:")
for word, count in freq.most_common(8):
    print(f"  {word:<14} {count}")

In [ ]:
# Common English stopwords to remove (function words carry little topical meaning)
STOPWORDS = {"it", "is", "a", "that", "in", "of", "the", "be", "or", "on", "his",
             "this", "so", "he", "to", "and", "such", "may", "must", "some", "one",
             "other", "their"}

# Functional cleaning pipeline: lowercase -> filter stopwords -> keep len > 2
cleaned = list(
    filter(lambda w: w not in STOPWORDS and len(w) > 2,
           map(str.lower, tokens))
)

print(f"Tokens before cleaning: {len(tokens)}")
print(f"Tokens after cleaning : {len(cleaned)}")
print("Content words:", cleaned)

In [ ]:
# Build a sorted vocabulary and a word -> id lookup (core NLP preprocessing step)
vocabulary = sorted(set(cleaned))
word2id = {word: idx for idx, word in enumerate(vocabulary)}
id2word = {idx: word for word, idx in word2id.items()}

print(f"Vocabulary size: {len(vocabulary)}")
print("First 10 vocab entries:", vocabulary[:10])
print("Encoded 'truth':", word2id.get("truth"))
print("Decoded id 0    :", id2word[0])

# Encode the cleaned corpus as a list of integer ids
encoded = [word2id[w] for w in cleaned]
print("Encoded sequence (first 12):", encoded[:12])

In [ ]:
# Bigrams = consecutive word pairs. zip(seq, seq[1:]) is the classic trick.
bigrams = [(a, b) for a, b in zip(cleaned, cleaned[1:])]
print(f"Total bigrams: {len(bigrams)}")

# Most frequent bigrams via Counter
bigram_freq = Counter(bigrams)
print("Top 5 bigrams:")
for pair, count in bigram_freq.most_common(5):
    print(f"  {pair} -> {count}")

## 9️⃣ Aplicação em Dataset Real — Penguins 🐧

Agora aplicamos as mesmas ferramentas funcionais a um dataset tabular real: **penguins** (do seaborn),
com medidas biométricas de três espécies de pinguins da Antártida.

In [ ]:
penguins = sns.load_dataset("penguins").dropna().reset_index(drop=True)
print(penguins.head())
print("\\nShape after dropna:", penguins.shape)

# Feature engineering with a lambda: bill ratio (length / depth)
penguins["bill_ratio"] = penguins.apply(
    lambda r: round(r["bill_length_mm"] / r["bill_depth_mm"], 3), axis=1
)

# Categorize body mass with a lambda
penguins["mass_class"] = penguins["body_mass_g"].apply(
    lambda g: "heavy" if g >= 4500 else ("medium" if g >= 3500 else "light")
)
print(penguins[["species", "bill_ratio", "body_mass_g", "mass_class"]].head())

In [ ]:
# map(): convert body mass from grams to kilograms
mass_kg = list(map(lambda g: round(g / 1000, 2), penguins["body_mass_g"]))
print("Body mass (kg), first 5:", mass_kg[:5])

# filter(): island names for penguins heavier than 5 kg
heavy_islands = list(
    filter(lambda pair: pair[1] > 5000,
           zip(penguins["island"], penguins["body_mass_g"]))
)
print(f"Penguins > 5kg: {len(heavy_islands)}")

# dict comprehension: mean bill ratio per species
mean_ratio = {
    sp: round(penguins.loc[penguins["species"] == sp, "bill_ratio"].mean(), 3)
    for sp in penguins["species"].unique()
}
print("Mean bill ratio per species:", mean_ratio)

## 🔟 Comparação de Desempenho — `for` vs comprehension vs `map`

Nem todo código funcional é automaticamente mais rápido, mas comprehensions e `map` frequentemente
superam o laço `for` explícito. Vamos medir com o módulo `timeit` (executa o trecho milhares de vezes).

In [ ]:
N = 10_000
setup = f"data = list(range({N}))"

def bench(stmt, number=200):
    '''Return average time in milliseconds for a statement.'''
    total = timeit.timeit(stmt, setup=setup, number=number)
    return total / number * 1000  # ms per run

for_loop = '''
out = []
for x in data:
    out.append(x * x)
'''
list_comp = "out = [x * x for x in data]"
map_call = "out = list(map(lambda x: x * x, data))"

print(f"{'for-loop':<18}: {bench(for_loop):.4f} ms")
print(f"{'list comprehension':<18}: {bench(list_comp):.4f} ms")
print(f"{'map + lambda':<18}: {bench(map_call):.4f} ms")
print("\\nNote: list comprehensions are usually the fastest for this kind of transform.")

## 📝 Resumo & Principais Aprendizados

- **Funções bem escritas** usam *type hints*, *docstrings* e, quando preciso, `*args`/`**kwargs`.
- **Lambdas** brilham como argumento de `sorted`, `map`, `filter` e `pandas.apply` — mas mantenha-as curtas.
- **`map`/`filter`/`reduce`** expressam transformações declarativas; lembre-se de materializar com `list()`.
- **Comprehensions** (list/dict/set) são concisas, legíveis e geralmente rápidas.
- **Generators** economizam memória — indispensáveis para grandes corpora e streams de dados.
- **`functools.partial`** pré-configura funções, ideal para pipelines reutilizáveis.
- **PLN na prática**: tokenização, frequência de palavras, limpeza, vocabulário e bigramas nascem
  naturalmente dessas ferramentas — a ponte perfeita entre **Linguística e Data Science**.

### 🔜 Próximos passos (Dia 5)
Programação Orientada a Objetos (OOP) para Data Science: classes, `@dataclass` e design de pipelines.

---
*Gutemberg Rapôso — Junior → Senior Data Scientist | Aberto a Oportunidades Remotas · Palmas-TO, Brasil*

In [ ]:
# 🎯 MINI-PROJECT: a reusable text-analysis pipeline combining every concept
# (functions, lambda, map, filter, reduce, comprehensions, Counter, partial, generators).

def analyze_text(text: str, stopwords: set = None, top_n: int = 5) -> dict:
    '''Run a compact NLP pipeline over raw text and return summary statistics.

    Steps: tokenize -> clean (map/filter) -> frequencies -> vocabulary -> bigrams.

    Parameters
    ----------
    text : str
        Raw input text.
    stopwords : set, optional
        Words to drop; defaults to an empty set.
    top_n : int
        How many top words/bigrams to report.

    Returns
    -------
    dict
        A report with counts, top words, vocabulary size and top bigrams.
    '''
    stopwords = stopwords or set()

    # 1. Tokenize with a comprehension over a regex
    toks = [w for w in re.findall(r"[a-z']+", text.lower())]

    # 2. Clean via map + filter + lambda
    content = list(filter(lambda w: w not in stopwords and len(w) > 2,
                          map(str.strip, toks)))

    # 3. Frequencies via Counter
    frequencies = Counter(content)

    # 4. Vocabulary via set comprehension + dict comprehension index
    vocab = {w: i for i, w in enumerate(sorted(set(content)))}

    # 5. Bigrams via zip + comprehension
    bg = Counter((a, b) for a, b in zip(content, content[1:]))

    # 6. Total characters via reduce (just to showcase the fold)
    total_chars = reduce(lambda acc, w: acc + len(w), content, 0)

    return {
        "n_tokens": len(toks),
        "n_content_words": len(content),
        "vocabulary_size": len(vocab),
        "avg_word_length": round(total_chars / len(content), 2) if content else 0,
        "top_words": frequencies.most_common(top_n),
        "top_bigrams": bg.most_common(top_n),
    }


# Pre-configure the pipeline for English using partial
analyze_english = partial(analyze_text, stopwords=STOPWORDS, top_n=5)

report = analyze_english(CORPUS)
print("=== TEXT ANALYSIS REPORT ===")
for key, value in report.items():
    print(f"{key:>16}: {value}")